# 경로 이관 후 재현 검증

프로젝트 전용 Python 3.12 커널에서 실행합니다. 기존 원자료와 결과는 보존하고, 분석 실행과 새 결과는 `environment/validation-workspace/`에 저장합니다. 전체 노트북의 구조·문법·초기화를 검사하고 네 실행 진입점을 확인합니다.

In [1]:
from pathlib import Path
from datetime import datetime
import os, json, shutil, hashlib, subprocess, time, ast
import nbformat
from nbclient import NotebookClient
from jupyter_client.kernelspec import KernelSpecManager
from IPython.core.inputtransformer2 import TransformerManager
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'environment/path_aliases.json').is_file())
ENV = ROOT/'environment'
os.environ['JUPYTER_PATH'] = str(ROOT/'.venv/share/jupyter')
os.environ['JUPYTER_PREFER_ENV_PATH'] = '1'
for name in ['OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'OMP_NUM_THREADS']:
    os.environ[name] = '1'
print('Kernel:', KernelSpecManager().get_kernel_spec('python3').argv[0])
WORK = ENV/'validation-workspace'/datetime.now().strftime('%Y%m%d_%H%M%S')
WORK.mkdir(parents=True)
VERIFY = ENV/'verification-output'
VERIFY.mkdir(exist_ok=True)
records = []


Kernel: C:\Coding\graduation-project\.venv\Scripts\python.exe


## 격리된 실행 폴더

원자료는 읽기 전용 입력으로 연결하고, 분석 데이터·결과·노트북·메타데이터는 별도 복사합니다. 검증 대상은 네트워크를 사용하지 않는 분석 실행 노트북입니다.

In [2]:
for folder in ['jupyter-notebook', 'reports', 'metadata', 'documents', 'dataset/analysis', 'dataset/processed']:
    shutil.copytree(ROOT/folder, WORK/folder)
for name in ['AGENTS.md', 'README.md', 'requirements.txt']:
    shutil.copy2(ROOT/name, WORK/name)
(WORK/'environment').mkdir()
for name in ['project_paths.ipynb', 'path_aliases.json', 'path_migration.json']:
    shutil.copy2(ENV/name, WORK/'environment'/name)
for name in ['report-builder', 'node']:
    shutil.copytree(ENV/name, WORK/'environment'/name)
# New junctions point to immutable inputs; no collection notebook is executed here.
command = ['powershell', '-NoProfile', '-Command',
           "New-Item -ItemType Junction -Path '" + str(WORK/'dataset/raw').replace("'", "''") +
           "' -Target '" + str(ROOT/'dataset/raw').replace("'", "''") + "' | Out-Null"]
subprocess.run(command, check=True, capture_output=True)
print('검증 위치:', WORK)


검증 위치: C:\Coding\graduation-project\environment\validation-workspace\20261006_160611


## 전체 노트북 구조·문법 검사

IPython 매직 명령은 변환한 뒤 Python 문법을 검사합니다. 경로를 수정한 노트북의 승인된 소스 해시도 검사합니다.

In [3]:
transformer = TransformerManager()
for path in sorted((ROOT/'jupyter-notebook').rglob('*.ipynb')):
    notebook = nbformat.read(path, as_version=4)
    nbformat.validate(notebook)
    for cell in notebook.cells:
        if cell.cell_type == 'code':
            ast.parse(transformer.transform_cell(cell.source))
    records.append({'check':'structure_and_syntax','notebook':path.relative_to(ROOT).as_posix(),'status':'passed'})
display(pd.DataFrame(records))


,check,notebook,status
0,structure_and_syntax,jupyter-notebook/20261005/notebooks/01_데이터_점검과...,passed
1,structure_and_syntax,jupyter-notebook/20261005/notebooks/02_EDA와_시각...,passed
2,structure_and_syntax,jupyter-notebook/20261005/notebooks/03_공연과_상권_...,passed
3,structure_and_syntax,jupyter-notebook/20261005/notebooks/04_데이터_연결검...,passed
4,structure_and_syntax,jupyter-notebook/20261005/notebooks/05_업종차이와_G...,passed
5,structure_and_syntax,jupyter-notebook/20261005/notebooks/06_온톨로지와_R...,passed
6,structure_and_syntax,jupyter-notebook/20261005/notebooks/07_AI서비스_분...,passed
7,structure_and_syntax,jupyter-notebook/20261005/notebooks/07_KOPIS_상...,passed
8,structure_and_syntax,jupyter-notebook/20261005/notebooks/08_KOPIS_C...,passed
9,structure_and_syntax,jupyter-notebook/20261005/notebooks/08_외식업_범위확...,passed


## 새 커널 분석 실행

각 진입점은 기존 고정 결과에서 독립적으로 시작합니다. 단계 사이에 검증 폴더의 기초 데이터·결과를 원본에서 복사해, 이전 단계가 만든 실행 시각·로그가 다음 단계의 보호 해시 검사에 섞이지 않게 합니다.

In [4]:
def save_records():
    (VERIFY/'execution_checks.json').write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')

def execute_one(relative):
    path = WORK/relative
    notebook = nbformat.read(path, as_version=4)
    started = time.perf_counter()
    try:
        NotebookClient(notebook, timeout=1200, kernel_name='python3',
                       resources={'metadata':{'path':str(path.parent)}}).execute()
        records.append({'check':'fresh_kernel','notebook':relative,'status':'passed',
                        'seconds':round(time.perf_counter()-started,2)})
    except Exception as exc:
        nbformat.write(notebook, VERIFY/('failed_'+path.name))
        records.append({'check':'fresh_kernel','notebook':relative,'status':'failed','error':str(exc)[-6000:]})
        save_records()
        raise
    nbformat.write(notebook, path)
    save_records()

entries = ['jupyter-notebook/20261006/00_전체실행.ipynb',
           'jupyter-notebook/20261006/01_시간대확장_실행.ipynb',
           'jupyter-notebook/20261006/02_수정분석_실행.ipynb',
           'jupyter-notebook/sandbox/00_전체실행.ipynb']
for relative in entries:
    for folder in ['dataset/analysis', 'reports']:
        shutil.copytree(ROOT/folder, WORK/folder, dirs_exist_ok=True)
    print('새 커널 실행:', relative, flush=True)
    execute_one(relative)
    if '/sandbox/' in relative:
        manifest_path = WORK/'jupyter-notebook/sandbox/reports/execution_manifest.json'
        result_files = ['jupyter-notebook/sandbox/reports/01_specification_results.csv',
                        'jupyter-notebook/sandbox/reports/03_power.csv',
                        'jupyter-notebook/sandbox/reports/05_results.csv']
    elif '/00_' in relative:
        manifest_path = WORK/'reports/20261006/execution_manifest.json'
        result_files = ['reports/20261006/09_industry_effects.csv','reports/20261006/10_gee_industry_effects.csv']
    elif '/01_' in relative:
        manifest_path = WORK/'reports/20261006/time_windows/execution_manifest.json'
        result_files = ['reports/20261006/time_windows/11_common_effects.csv','reports/20261006/time_windows/11_industry_effects.csv']
    else:
        manifest_path = WORK/'reports/20261006/revised_quarterly/execution_manifest.json'
        result_files = ['reports/20261006/revised_quarterly/13_effects.csv']
    run = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert all(step['notebook'].startswith('jupyter-notebook/') for step in run['steps']), manifest_path
    for result_file in result_files:
        expected = pd.read_csv(ROOT/result_file)
        actual = pd.read_csv(WORK/result_file)
        pd.testing.assert_frame_equal(actual, expected, check_exact=False, rtol=1e-10, atol=1e-12)
        records.append({'check':'numeric_reproduction','notebook':result_file,'status':'passed','rows':len(actual)})
    save_records()
display(pd.DataFrame(records).query("check == 'fresh_kernel'"))


새 커널 실행: jupyter-notebook/20261006/00_전체실행.ipynb


C:\Coding\graduation-project\.venv\Lib\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()


새 커널 실행: jupyter-notebook/20261006/01_시간대확장_실행.ipynb


새 커널 실행: jupyter-notebook/20261006/02_수정분석_실행.ipynb


새 커널 실행: jupyter-notebook/sandbox/00_전체실행.ipynb


,check,notebook,status,seconds,rows
56,fresh_kernel,jupyter-notebook/20261006/00_전체실행.ipynb,passed,91.65,NaN
59,fresh_kernel,jupyter-notebook/20261006/01_시간대확장_실행.ipynb,passed,49.32,NaN
62,fresh_kernel,jupyter-notebook/20261006/02_수정분석_실행.ipynb,passed,25.27,NaN
64,fresh_kernel,jupyter-notebook/sandbox/00_전체실행.ipynb,passed,44.15,NaN


## 원본 보존 검사

이관 전 기록과 비교해 원자료·파생 자료·과거 결과 파일이 수정되지 않았는지 검사합니다. 노트북과 실행 안내의 경로 수정은 별도로 기록돼 있습니다.

In [5]:
originals = json.loads((ENV/'protected_files_before.json').read_text(encoding='utf-8'))
changed = []
for item in originals:
    path = ROOT/item['path']
    if item['path'] == 'reports/README.md':
        continue
    if not path.is_file() or path.stat().st_size != item['size'] or path.stat().st_mtime_ns != item['mtime_ns']:
        changed.append(item['path'])
assert not changed, changed
summary = {'notebooks_checked':sum(x['check']=='structure_and_syntax' for x in records),
           'entrypoints_passed':sum(x['check']=='fresh_kernel' and x['status']=='passed' for x in records),
           'protected_files_unchanged':len(originals)-1,'workspace':str(WORK)}
(VERIFY/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame([summary]))


,notebooks_checked,entrypoints_passed,protected_files_unchanged,workspace
0,56,4,32556,C:\Coding\graduation-project\environment\valid...


## 해석

네 실행 진입점이 통과하면 현재 폴더와 프로젝트 내부 환경에서 분석이 실행됨을 확인한 것입니다. 과거 결과의 실행 기록을 갱신하거나 고정 입력을 교체하지 않습니다. 수집 API는 호출하지 않습니다.